In [ ]:
from pathlib import Path
import os

# Set PROJECT_DATA_DIR before launching Jupyter to use data stored elsewhere.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".gitignore").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = Path(os.environ.get("PROJECT_DATA_DIR", str(PROJECT_ROOT / "data"))).expanduser().resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
#Best ALL

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    balanced_accuracy_score, roc_auc_score,
    classification_report, confusion_matrix
)
import seaborn as sns
import matplotlib.pyplot as plt
import shap

df = pd.read_csv(str(DATA_DIR / 'AI_UNITE_CRSS_FARS_merged_subset_same_features.csv'),
                 encoding='latin-1')

df = df.dropna(subset=['INJ_SEV'])
df['INJ_SEV'] = df['INJ_SEV'].astype(int)

target = 'INJ_SEV'

categorical_cols = [
    'PEDS','PERNOTMVIT','VE_TOTAL','PVH_INVL','PERMVIT','MONTH','DAY_WEEK','YEAR',
    'HARM_EV','MAN_COLL','TYP_INT','REL_ROAD','WRK_ZONE','LGT_COND','WEATHER',
    'DRDISTRACT','DRIMPAIR','PER_NO','SPEC_USE','AGE','SEX','REST_USE','REST_MIS',
    'HELM_USE','HELM_MIS','DRINKING','ALC_STATUS','ATST_TYP',
    'ALC_RES','DRUGS','STR_VEH','LOCATION','VE_FORMS','HIT_RUN','BODY_TYP','TOW_VEH',
    'CARGO_BT','HAZ_INV','EMER_USE','DR_PRES','SPEEDREL','VTRAFWAY','VSPD_LIM',
    'VSURCOND','VISION' # , 'AIR_BAG','EJECTION',
] 
numeric_cols = ['TRAV_SP']

for col in categorical_cols:
    df[col] = df[col].astype('category')

X = df[categorical_cols + numeric_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)


params = {
    "objective": "multi:softprob",
    "num_class": 4,
    "tree_method": "hist",  
    "device": "cuda",
    "eval_metric": ["mlogloss","auc","merror"],
    "seed": 42
}

dtrain = xgb.DMatrix(X_train, y_train, enable_categorical=True)
dtest  = xgb.DMatrix(X_test,  y_test,  enable_categorical=True)

cv = xgb.cv(
    params, dtrain,
    num_boost_round=1000,
    nfold=5,
    early_stopping_rounds=30,
    verbose_eval=False
)

best_n = int(cv['test-mlogloss-mean'].idxmin()+1)

model = xgb.train(params, dtrain, num_boost_round=best_n)

y_prob = model.predict(dtest)
y_pred = np.argmax(y_prob, axis=1)

print("Balanced accuracy:", round(balanced_accuracy_score(y_test, y_pred), 4))
print("Test AUROC:", round(roc_auc_score(
      y_test, y_prob, multi_class='ovo'), 4))
print(classification_report(y_test, y_pred, digits=3))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'Lvl{i}' for i in range(4)],
            yticklabels=[f'Lvl{i}' for i in range(4)])
plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout(); plt.show()




In [ ]:
sample_size = min(10000, len(X_train))
shap_sample = X_train.sample(sample_size, random_state=42)


shap_sample_numeric = shap_sample.copy()
for col in categorical_cols:
	shap_sample_numeric[col] = shap_sample_numeric[col].cat.codes

explainer = shap.TreeExplainer(model)


shap_values = explainer.shap_values(shap_sample_numeric, check_additivity=False)


import numpy as np
mean_abs = np.abs(shap_values).mean(axis=(0, 2))
importance = pd.Series(mean_abs, index=model.feature_names).sort_values(ascending=False)


shap.summary_plot(shap_values, shap_sample, plot_type="bar", max_display=20)


In [ ]:
#Best ALL

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    balanced_accuracy_score, roc_auc_score,
    classification_report, confusion_matrix
)
import seaborn as sns
import matplotlib.pyplot as plt
import shap

df = pd.read_csv(str(DATA_DIR / 'AI_UNITE_CRSS_FARS_merged_subset_same_features.csv'),
                 encoding='latin-1')

df = df.dropna(subset=['INJ_SEV'])
df['INJ_SEV'] = df['INJ_SEV'].astype(int)

target = 'INJ_SEV'

categorical_cols = [
    'PERNOTMVIT','PVH_INVL','PERMVIT','MONTH','DAY_WEEK','YEAR',
    'HARM_EV','MAN_COLL','TYP_INT','REL_ROAD','WRK_ZONE','LGT_COND','WEATHER',
    'DRDISTRACT','DRIMPAIR','SPEC_USE','SEX','REST_USE','REST_MIS',
    'HELM_USE','HELM_MIS','DRINKING','ALC_STATUS','ATST_TYP',
    'ALC_RES','DRUGS','STR_VEH','LOCATION','VE_FORMS','HIT_RUN','BODY_TYP','TOW_VEH',
    'CARGO_BT','HAZ_INV','EMER_USE','DR_PRES','SPEEDREL','VTRAFWAY',
    'VSURCOND','VISION' # , 'AIR_BAG','EJECTION',
] 
numeric_cols = ['TRAV_SP', 'AGE', 'VSPD_LIM', 'VE_TOTAL', 'PEDS',]

for col in categorical_cols:
    df[col] = df[col].astype('category')

X = df[categorical_cols + numeric_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

params = {
    "objective": "multi:softprob",
    "num_class": 4,
    "tree_method": "hist",  
    "device": "cuda",
    "eval_metric": ["mlogloss","auc","merror"],
    "seed": 42
}

dtrain = xgb.DMatrix(X_train, y_train, enable_categorical=True)
dtest  = xgb.DMatrix(X_test,  y_test,  enable_categorical=True)

cv = xgb.cv(
    params, dtrain,
    num_boost_round=1000,
    nfold=5,
    early_stopping_rounds=30,
    verbose_eval=False
)

best_n = int(cv['test-mlogloss-mean'].idxmin()+1)

model = xgb.train(params, dtrain, num_boost_round=best_n)

y_prob = model.predict(dtest)
y_pred = np.argmax(y_prob, axis=1)

print("Balanced accuracy:", round(balanced_accuracy_score(y_test, y_pred), 4))
print("Test AUROC:", round(roc_auc_score(
      y_test, y_prob, multi_class='ovo'), 4))
print(classification_report(y_test, y_pred, digits=3))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'Lvl{i}' for i in range(4)],
            yticklabels=[f'Lvl{i}' for i in range(4)])
plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout(); plt.show()




In [ ]:
sample_size = min(10000, len(X_train))
shap_sample = X_train.sample(sample_size, random_state=42)


shap_sample_numeric = shap_sample.copy()
for col in categorical_cols:
	shap_sample_numeric[col] = shap_sample_numeric[col].cat.codes

explainer = shap.TreeExplainer(model)


shap_values = explainer.shap_values(shap_sample_numeric, check_additivity=False)


import numpy as np
mean_abs = np.abs(shap_values).mean(axis=(0, 2))
importance = pd.Series(mean_abs, index=model.feature_names).sort_values(ascending=False)


shap.summary_plot(shap_values, shap_sample, plot_type="bar", max_display=20)


In [ ]:
#Best Impairments ONLY

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    balanced_accuracy_score, roc_auc_score,
    classification_report, confusion_matrix
)
import seaborn as sns
import matplotlib.pyplot as plt
import shap

# Load 
df = pd.read_csv(str(DATA_DIR / 'AI_UNITE_CRSS_FARS_merged_IMPAIRMENT_ONLY.csv'),
                 encoding='latin-1')

df = df.dropna(subset=['INJ_SEV'])
df['INJ_SEV'] = df['INJ_SEV'].astype(int)

target = 'INJ_SEV'

categorical_cols = [
    'PERNOTMVIT','PVH_INVL','PERMVIT','MONTH','DAY_WEEK','YEAR',
    'HARM_EV','MAN_COLL','TYP_INT','REL_ROAD','WRK_ZONE','LGT_COND','WEATHER',
    'DRDISTRACT','DRIMPAIR','SPEC_USE','SEX','REST_USE','REST_MIS',
    'HELM_USE','HELM_MIS','DRINKING','ALC_STATUS',
    'ALC_RES','DRUGS','STR_VEH','LOCATION','VE_FORMS','HIT_RUN','BODY_TYP','TOW_VEH',
    'CARGO_BT','HAZ_INV','EMER_USE','DR_PRES','SPEEDREL','VTRAFWAY',
    'VSURCOND','VISION', 'VSPD_LIM', 'VE_TOTAL', 'PEDS' # , 'AIR_BAG','EJECTION','ATST_TYP',
] 
numeric_cols = ['TRAV_SP', 'AGE']

for col in categorical_cols:
    df[col] = df[col].astype('category')

X = df[categorical_cols + numeric_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

params = {
    "objective": "multi:softprob",
    "num_class": 4,
    "tree_method": "hist",  
    "device": "cuda",
    "eval_metric": ["mlogloss","auc","merror"],
    "seed": 42
}

dtrain = xgb.DMatrix(X_train, y_train, enable_categorical=True)
dtest  = xgb.DMatrix(X_test,  y_test,  enable_categorical=True)

cv = xgb.cv(
    params, dtrain,
    num_boost_round=1000,
    nfold=5,
    early_stopping_rounds=30,
    verbose_eval=False
)

best_n = int(cv['test-mlogloss-mean'].idxmin()+1)

model = xgb.train(params, dtrain, num_boost_round=best_n)

y_prob = model.predict(dtest)
y_pred = np.argmax(y_prob, axis=1)

print("Balanced accuracy:", round(balanced_accuracy_score(y_test, y_pred), 4))
print("Test AUROC:", round(roc_auc_score(
      y_test, y_prob, multi_class='ovo'), 4))
print(classification_report(y_test, y_pred, digits=3))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'Lvl{i}' for i in range(4)],
            yticklabels=[f'Lvl{i}' for i in range(4)])
plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout(); plt.show()




In [ ]:

sample_size = min(10000, len(X_train))
shap_sample = X_train.sample(sample_size, random_state=42)


shap_sample_numeric = shap_sample.copy()
for col in categorical_cols:
	shap_sample_numeric[col] = shap_sample_numeric[col].cat.codes

explainer = shap.TreeExplainer(model)


shap_values = explainer.shap_values(shap_sample_numeric, check_additivity=False)


import numpy as np
mean_abs = np.abs(shap_values).mean(axis=(0, 2))
importance = pd.Series(mean_abs, index=model.feature_names).sort_values(ascending=False)


shap.summary_plot(shap_values, shap_sample, plot_type="bar", max_display=20)
